# Overview

There are four possible providers of compute services, or consulting services on campus:

- [Cornell Center of Social Sciences (CCSS)](https://socialsciences.cornell.edu/computing-and-data/cloud-computing-solutions) offers general purpose and secure computing services.
- [Center for Advanced Computing (CAC)](https://www.cac.cornell.edu/about/Default.aspx) offers advanced and cloud computing solutions, including in the [national (ACCESS-CI)](https://access-ci.org/) and [state (NYSTAR)](http://esd.ny.gov/nystar/) computing infrastructure 
- [Biotechnology Resource Center (BioHPC)](https://www.biotech.cornell.edu/facilities-brc) offers advanced on-campus computing solutions (including managing the Econ Department's own cluster)
- [Cornell Information Technology (CIT)](https://it.cornell.edu/) is the general Cornell IT department, but can also engineer solutions for cloud services (Cornell has framework subscriptions to [Amazon Web Service (AWS/EC2)](https://it.cornell.edu/cornell-cloud/amazon-web-services-aws-contract) and [Microsoft cloud services (Azure)](https://it.cornell.edu/cornell-cloud/cornell-azure-contract)).


In [ ]:
import pandas as pd
import os
from IPython.display import HTML


def find_project_root(start_dir='.', file_to_find='favicon.ico'):
    """
    Find the project root directory by searching for a specific file.
    
    Args:
        start_dir (str): The directory to start the search from.
        file_to_find (str): The name of the file to search for.
        
    Returns:
        str: The absolute path of the project root directory.
    """
    current_dir = os.path.abspath(start_dir)
    
    while True:
        # Check if the file exists in the current directory
        if os.path.isfile(os.path.join(current_dir, file_to_find)):
            return current_dir
        
        # Check if we've reached the root directory
        parent_dir = os.path.dirname(current_dir)
        if parent_dir == current_dir:
            raise FileNotFoundError(f"Could not find '{file_to_find}' in the directory tree.")
        
        # Move up one directory
        current_dir = parent_dir

# Example usage
project_root = find_project_root()
#print(f"Project root directory: {project_root}")



# Read the CSV file
df = pd.read_csv(os.path.join(project_root,"_data","eccochoice.csv"))

# Compute the BioHPC ECCO row from the actual SLURM node list
# (same logic as biohpc/slurm.md: flex + slurm nodes, HT doubles cores)
nodes = pd.read_csv(os.path.join(project_root, "_data", "ecconodes.csv"))
nodes = nodes[nodes['allocation'].str.contains('flex|slurm', na=False)]
nodes['cores'] = nodes['cores per CPU'] * nodes['CPUs']
nodes.loc[nodes['HT'] == True, 'cores'] *= 2
ecco = df['System'] == 'BioHPC ECCO'
# maxNodeRAM and maxNodecores are independent maxima: the most RAM on any
# single node, and the most cores on any single node (possibly a different node)
df.loc[ecco, 'maxNodeRAM'] = f"{nodes['RAM'].max()}GB"
df.loc[ecco, 'maxNodecores'] = str(nodes['cores'].max())
df.loc[ecco, 'totalRAM'] = f"up to {nodes['RAM'].sum()}GB"
df.loc[ecco, 'totalcores'] = f"up to {nodes['cores'].sum()}"

# Compute the BioHPC bioslurm row from the bioslurm hardware table
# (transcribed from https://biohpc.cornell.edu/lab/userguide.aspx?a=bioslurm),
# excluding nodes that are only in the workshop-only 'training' partition
bsl = pd.read_csv(os.path.join(project_root, "_data", "bioslurmnodes.csv"))
bsl = bsl[bsl['partition'] != 'training']
bsl['RAM'] = (bsl['RAM (MB)'] / 1024).round().astype(int)
bioslurm = df['System'] == 'BioHPC `bioslurm`'
df.loc[bioslurm, 'maxNodeRAM'] = f"{bsl['RAM'].max()}GB"
df.loc[bioslurm, 'maxNodecores'] = str(bsl['Slurm CPUs'].max())
df.loc[bioslurm, 'totalRAM'] = f"{bsl['RAM'].sum()}GB"
df.loc[bioslurm, 'totalcores'] = str(bsl['Slurm CPUs'].sum())
df.loc[bioslurm, 'goodfor'] = f"HPC + {bsl['GPUs'].sum()} GPUs"
df.loc[bioslurm, 'GPU'] = f"Yes ({bsl['GPUs'].sum()} GPUs on {(bsl['GPUs'] > 0).sum()} nodes)"
bioslurm_summary = (f"The `bioslurm` cluster has {len(bsl)} nodes available to general jobs, "
                    f"with {bsl['Slurm CPUs'].min()} to {bsl['Slurm CPUs'].max()} CPUs "
                    f"and {bsl['RAM'].min()} to {bsl['RAM'].max()} GB of RAM per node, "
                    f"and {bsl['GPUs'].sum()} GPUs on {(bsl['GPUs'] > 0).sum()} nodes "
                    f"(see the [bioslurm user guide](https://biohpc.cornell.edu/lab/userguide.aspx?a=bioslurm)).")

# Render `code` in system names
df['System'] = df['System'].str.replace(r'`([^`]+)`', r'<code>\1</code>', regex=True)

# Create a new column with the hyperlink
df['Further Info Link'] = "<a href='" + df['Further info URL'] + "' target='_blank'>" + df['Further info'] + "</a>"

# Drop the original columns if you don't need them in the table
df = df.drop(['Further info', 'Further info URL'], axis=1)

# Clearer column labels for readers
df = df.rename(columns={'maxNodeRAM': 'Max RAM per node',
                        'maxNodecores': 'Max cores per node',
                        'totalRAM': 'Total RAM',
                        'totalcores': 'Total cores'})

# Convert the DataFrame to an HTML table
table = df.to_html(index=False, 
                   classes='table table-striped table-bordered table-sm', escape=False, 
                   render_links=True)


# Render the HTML table in Jupyter Notebook
from IPython.display import Markdown, display
display(HTML(table))
display(Markdown(bioslurm_summary))


*Max RAM per node* and *Max cores per node* are independent: each is the largest value found on any single node, and the two may come from different nodes. Core counts are what the scheduler can allocate, so they include hyper-threads where enabled. For the BioHPC clusters, they are computed from the node lists ([ECCO](biohpc/slurm.md#fulltable), [bioslurm](https://biohpc.cornell.edu/lab/userguide.aspx?a=bioslurm)).

The table is sorted by increasing complexity of preparation and cost (which are collinear). A starting point on how to easily use the latter cloud resources in a (relatively) easy way are

- the nice [tutorial by Andrew Heiss](https://www.andrewheiss.com/blog/2018/07/30/disposable-supercomputer-future/) 
- the [analogsea](https://github.com/pachadotdev/analogsea) and [future](https://cran.r-project.org/web/packages/future/index.html) R packages.